# Phase 1 — Exploratory Data Analysis (EDA)
## XAI-Based Network Intrusion Detection System

**Dataset:** CICIDS-2017 (Canadian Institute for Cybersecurity)  
**Author:** Chandra Sekhar Chakraborty  
**Goal:** Understand the dataset structure, class distribution, feature statistics, and correlations before preprocessing.

---

### EDA Roadmap
1. [Setup & Imports](#1-setup--imports)
2. [Load Dataset](#2-load-dataset)
3. [Basic Dataset Info](#3-basic-dataset-info)
4. [Class Distribution](#4-class-distribution)
5. [Missing Values & Infinity Check](#5-missing-values--infinity-check)
6. [Feature Statistics](#6-feature-statistics)
7. [Correlation Heatmap](#7-correlation-heatmap)
8. [Top Feature Analysis](#8-top-feature-analysis)
9. [Attack-wise Feature Comparison](#9-attack-wise-feature-comparison)
10. [Key EDA Findings & Next Steps](#10-key-eda-findings--next-steps)

## 1. Setup & Imports

In [ ]:
import os
import glob
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
plt.rcParams.update({
    'figure.facecolor': '#1a1a2e',
    'axes.facecolor': '#16213e',
    'axes.edgecolor': '#0f3460',
    'axes.labelcolor': '#e0e0e0',
    'xtick.color': '#e0e0e0',
    'ytick.color': '#e0e0e0',
    'text.color': '#e0e0e0',
    'grid.color': '#0f3460',
    'font.family': 'monospace',
    'figure.dpi': 120
})

# Color palette for attack classes
ATTACK_COLORS = [
    '#00d4ff', '#ff6b6b', '#ffd93d', '#6bcb77', '#4d96ff',
    '#ff9f43', '#a29bfe', '#fd79a8', '#55efc4', '#fdcb6e',
    '#e17055', '#74b9ff', '#00cec9', '#b2bec3', '#dfe6e9'
]

print('✅ Libraries loaded successfully')
print(f'   NumPy  : {np.__version__}')
print(f'   Pandas : {pd.__version__}')
print(f'   Seaborn: {sns.__version__}')

## 2. Load Dataset

> **Note:** Place all CICIDS-2017 CSV files in `../data/raw/`. The cell below auto-detects and concatenates all day files.
> If running a quick test, set `SAMPLE_MODE = True` to load only the first 100K rows per file.

In [ ]:
DATA_DIR    = '../data/raw/'
SAMPLE_MODE = False
SAMPLE_ROWS = 100_000
RANDOM_SEED = 42

csv_files = sorted(glob.glob(os.path.join(DATA_DIR, '*.csv')))

if not csv_files:
    raise FileNotFoundError(
        f'No CSV files found in {DATA_DIR}.\n'
        'Download CICIDS-2017 from https://www.unb.ca/cic/datasets/ids-2017.html '
        'and place the CSV files in ../data/raw/'
    )

print(f'📂 Found {len(csv_files)} CSV file(s):')
for f in csv_files:
    size_mb = os.path.getsize(f) / (1024 ** 2)
    print(f'   {os.path.basename(f):50s}  {size_mb:7.1f} MB')

dfs = []
for f in csv_files:
    _df = pd.read_csv(f, low_memory=False, nrows=SAMPLE_ROWS if SAMPLE_MODE else None)
    dfs.append(_df)
    print(f'   Loaded: {os.path.basename(f)} → {len(_df):,} rows')

df = pd.concat(dfs, ignore_index=True)
df.columns = df.columns.str.strip()
print(f'\n✅ Combined dataset: {df.shape[0]:,} rows × {df.shape[1]} columns')

## 3. Basic Dataset Info

In [ ]:
print('=' * 60)
print('DATASET OVERVIEW')
print('=' * 60)
print(f'  Shape        : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'  Memory usage : {df.memory_usage(deep=True).sum() / (1024**2):.1f} MB')
print(f'  Label column : {"Label" if "Label" in df.columns else "Not found"}')
print()
dtype_summary = df.dtypes.value_counts()
print('  Data types:')
for dtype, count in dtype_summary.items():
    print(f'    {str(dtype):15s}: {count} columns')
print('\n  First 5 rows:')
df.head()

In [ ]:
LABEL_COL = 'Label'
if LABEL_COL not in df.columns:
    candidates = [c for c in df.columns if 'label' in c.lower()]
    if candidates:
        LABEL_COL = candidates[0]
        print(f'ℹ️  Label column detected as: "{LABEL_COL}"')
    else:
        raise ValueError('Could not find the label column.')

FEATURE_COLS = [c for c in df.columns if c != LABEL_COL]
print(f'\n  Total features  : {len(FEATURE_COLS)}')
print(f'  Label column    : "{LABEL_COL}"')
print(f'\n  Feature columns (first 10): {FEATURE_COLS[:10]}')

## 4. Class Distribution

> Understanding class imbalance is critical — it drives the SMOTE strategy in Phase 2.

In [ ]:
class_counts = df[LABEL_COL].value_counts().reset_index()
class_counts.columns = ['Attack_Class', 'Count']
class_counts['Percentage'] = (class_counts['Count'] / len(df) * 100).round(3)

print('CLASS DISTRIBUTION')
print('-' * 55)
for _, row in class_counts.iterrows():
    bar = '█' * int(row['Percentage'] / 2)
    print(f'  {row["Attack_Class"]:40s} {row["Count"]:>8,}  ({row["Percentage"]:6.2f}%)  {bar}')

print(f'\n  Total classes : {len(class_counts)}')
print(f'  Total samples : {len(df):,}')
imbalance_ratio = class_counts['Count'].max() / class_counts['Count'].min()
print(f'  Imbalance ratio (max/min): {imbalance_ratio:,.0f}:1  → SMOTE required in Phase 2')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('CICIDS-2017 — Class Distribution', fontsize=14, fontweight='bold', color='#00d4ff')

ax = axes[0]
bars = ax.barh(class_counts['Attack_Class'], class_counts['Count'],
               color=ATTACK_COLORS[:len(class_counts)], edgecolor='none', height=0.65)
ax.set_xlabel('Sample Count', fontsize=10)
ax.set_title('Sample Count per Class', fontsize=11, color='#e0e0e0')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for bar, (_, row) in zip(bars, class_counts.iterrows()):
    ax.text(bar.get_width() * 1.01, bar.get_y() + bar.get_height() / 2,
            f'{row["Percentage"]:.1f}%', va='center', ha='left', fontsize=8, color='#b2bec3')
ax.invert_yaxis()

ax2 = axes[1]
ax2.barh(class_counts['Attack_Class'], class_counts['Count'],
         color=ATTACK_COLORS[:len(class_counts)], edgecolor='none', height=0.65)
ax2.set_xscale('log')
ax2.set_xlabel('Sample Count (log scale)', fontsize=10)
ax2.set_title('Log Scale — Reveals Minority Classes', fontsize=11, color='#e0e0e0')
ax2.invert_yaxis()
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
os.makedirs('../docs/eda_plots', exist_ok=True)
plt.savefig('../docs/eda_plots/01_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved → ../docs/eda_plots/01_class_distribution.png')

## 5. Missing Values & Infinity Check

> CICIDS-2017 is known to have `Inf` and `NaN` values in certain flow-rate features. We identify them here — the cleaning strategy is implemented in Phase 2.

In [ ]:
null_counts = df[FEATURE_COLS].isnull().sum()
null_cols = null_counts[null_counts > 0].sort_values(ascending=False)

print('NULL / NaN VALUES')
print('-' * 50)
if len(null_cols) == 0:
    print('  ✅ No NaN values found in feature columns')
else:
    for col, cnt in null_cols.items():
        print(f'  {col:45s}: {cnt:,} NaN  ({cnt/len(df)*100:.3f}%)')

print('\nINFINITY VALUES')
print('-' * 50)
numeric_cols = df[FEATURE_COLS].select_dtypes(include=[np.number]).columns.tolist()
inf_counts = {}
for col in numeric_cols:
    n_inf = np.isinf(df[col]).sum()
    if n_inf > 0:
        inf_counts[col] = n_inf

if not inf_counts:
    print('  ✅ No Inf values found in numeric columns')
else:
    for col, cnt in sorted(inf_counts.items(), key=lambda x: -x[1]):
        print(f'  {col:45s}: {cnt:,} Inf  ({cnt/len(df)*100:.3f}%)')

print(f'\n📋 Phase 2 Cleaning Plan:')
print(f'   → Replace Inf with column max (per class)')
print(f'   → Drop rows with NaN after Inf replacement')
print(f'   → Columns with >30% NaN will be dropped entirely')

## 6. Feature Statistics

> Descriptive statistics help identify skewed features, zero-variance columns, and outlier ranges.

In [ ]:
df_stats = df[numeric_cols].replace([np.inf, -np.inf], np.nan)

stats = df_stats.describe().T
stats['skewness'] = df_stats.skew()
stats['kurtosis'] = df_stats.kurtosis()
stats['zero_pct'] = (df_stats == 0).mean() * 100
stats['cv'] = (stats['std'] / stats['mean'].abs()).replace([np.inf, -np.inf], np.nan)

zero_var = stats[stats['std'] == 0]
high_skew = stats[stats['skewness'].abs() > 10].sort_values('skewness', ascending=False)
high_zero = stats[stats['zero_pct'] > 50].sort_values('zero_pct', ascending=False)

print(f'FEATURE STATISTICS SUMMARY')
print(f'  Total numeric features    : {len(numeric_cols)}')
print(f'  Zero-variance features    : {len(zero_var)}')
print(f'  High skewness (|skew|>10) : {len(high_skew)}')
print(f'  High zero-pct (>50%)      : {len(high_zero)}')

if len(zero_var) > 0:
    print(f'\n  ⚠️  Zero-variance columns (will be dropped in Phase 2):')
    for col in zero_var.index:
        print(f'     - {col}')

print(f'\n  Top 10 most skewed features:')
print(high_skew[['mean', 'std', 'skewness', 'zero_pct']].head(10).to_string())

In [ ]:
top_var_features = stats.nlargest(9, 'cv').index.tolist()

fig, axes = plt.subplots(3, 3, figsize=(15, 10))
fig.suptitle('Feature Distributions — Top 9 by Coefficient of Variation',
             fontsize=13, fontweight='bold', color='#00d4ff')

for ax, feat in zip(axes.flatten(), top_var_features):
    data = df_stats[feat].dropna()
    clip_max = data.quantile(0.99)
    data_clipped = data.clip(upper=clip_max)
    ax.hist(data_clipped, bins=50, color='#00d4ff', edgecolor='none', alpha=0.8)
    ax.set_title(feat[:35], fontsize=8, color='#e0e0e0')
    ax.set_xlabel('Value (clipped at 99th pct)', fontsize=7)
    ax.set_ylabel('Count', fontsize=7)
    ax.tick_params(labelsize=7)

plt.tight_layout()
plt.savefig('../docs/eda_plots/02_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved → ../docs/eda_plots/02_feature_distributions.png')

## 7. Correlation Heatmap

> We identify highly correlated feature pairs (|r| > 0.9) that are candidates for removal in Phase 2 to reduce redundancy.

In [ ]:
CORR_SAMPLE = 50_000
df_corr = df_stats.sample(min(CORR_SAMPLE, len(df_stats)), random_state=RANDOM_SEED)
corr_matrix = df_corr.corr()

upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_pairs = (
    upper_tri.stack()
    .reset_index()
    .rename(columns={'level_0': 'Feature_A', 'level_1': 'Feature_B', 0: 'Correlation'})
)
high_corr_pairs = high_corr_pairs[
    high_corr_pairs['Correlation'].abs() > 0.9
].sort_values('Correlation', ascending=False)

print(f'HIGH CORRELATION PAIRS (|r| > 0.90)')
print(f'  Found {len(high_corr_pairs)} pairs')
print()
print(high_corr_pairs.head(20).to_string(index=False))

In [ ]:
top25 = stats.nlargest(25, 'std').index.tolist()
corr_top25 = df_corr[top25].corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_top25, dtype=bool))
cmap = sns.diverging_palette(240, 10, as_cmap=True)

sns.heatmap(
    corr_top25, mask=mask, cmap=cmap,
    vmin=-1, vmax=1, center=0,
    annot=False, linewidths=0.3, linecolor='#1a1a2e',
    square=True, ax=ax,
    cbar_kws={'shrink': 0.8, 'label': 'Pearson r'}
)

ax.set_title('Feature Correlation Heatmap — Top 25 Features by Variance',
             fontsize=13, fontweight='bold', color='#00d4ff', pad=15)
ax.tick_params(axis='x', rotation=90, labelsize=7)
ax.tick_params(axis='y', rotation=0, labelsize=7)
plt.tight_layout()
plt.savefig('../docs/eda_plots/03_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved → ../docs/eda_plots/03_correlation_heatmap.png')

## 8. Top Feature Analysis

> Using variance and inter-class separation to identify the most discriminative features.

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier

QUICK_SAMPLE = 30_000
df_quick = df.sample(min(QUICK_SAMPLE, len(df)), random_state=RANDOM_SEED).copy()

X_quick = df_quick[numeric_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
le = LabelEncoder()
y_quick = le.fit_transform(df_quick[LABEL_COL])

print('⏳ Training quick Random Forest for feature importance (EDA only)...')
rf_quick = RandomForestClassifier(n_estimators=50, max_depth=8, n_jobs=-1, random_state=RANDOM_SEED)
rf_quick.fit(X_quick, y_quick)

importances = pd.Series(rf_quick.feature_importances_, index=numeric_cols)
top_features = importances.sort_values(ascending=False).head(20)

print('\nTOP 20 FEATURES BY RANDOM FOREST IMPORTANCE (EDA quick pass):')
print('-' * 55)
for feat, imp in top_features.items():
    bar = '█' * int(imp * 500)
    print(f'  {feat:45s}: {imp:.5f}  {bar}')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
colors = [ATTACK_COLORS[i % len(ATTACK_COLORS)] for i in range(len(top_features))]
bars = ax.barh(top_features.index[::-1], top_features.values[::-1],
               color=colors[::-1], edgecolor='none', height=0.65)
ax.set_xlabel('Feature Importance Score', fontsize=10)
ax.set_title('Top 20 Features — Random Forest Importance (EDA Quick Pass)',
             fontsize=11, color='#00d4ff', fontweight='bold')
for bar in bars:
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height() / 2,
            f'{bar.get_width():.4f}', va='center', ha='left', fontsize=8, color='#b2bec3')
ax.set_xlim(0, top_features.values.max() * 1.2)
plt.tight_layout()
plt.savefig('../docs/eda_plots/04_feature_importance_eda.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved → ../docs/eda_plots/04_feature_importance_eda.png')

## 9. Attack-wise Feature Comparison

> Box plots comparing the top 4 features across attack classes.

In [ ]:
TOP4 = top_features.head(4).index.tolist()
df_plot = df.sample(min(20_000, len(df)), random_state=RANDOM_SEED).copy()
df_plot[TOP4] = df_plot[TOP4].replace([np.inf, -np.inf], np.nan)
for feat in TOP4:
    clip_max = df_plot[feat].quantile(0.99)
    df_plot[feat] = df_plot[feat].clip(upper=clip_max)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Attack-wise Distribution — Top 4 Discriminative Features',
             fontsize=13, fontweight='bold', color='#00d4ff')

unique_classes = df_plot[LABEL_COL].unique()
class_color_map = {cls: ATTACK_COLORS[i % len(ATTACK_COLORS)] for i, cls in enumerate(sorted(unique_classes))}

for ax, feat in zip(axes.flatten(), TOP4):
    groups = [df_plot[df_plot[LABEL_COL] == cls][feat].dropna() for cls in sorted(unique_classes)]
    bp = ax.boxplot(groups, patch_artist=True, notch=False, showfliers=False,
                    medianprops=dict(color='#ffffff', linewidth=1.5))
    for patch, cls in zip(bp['boxes'], sorted(unique_classes)):
        patch.set_facecolor(class_color_map[cls])
        patch.set_alpha(0.75)
    ax.set_xticklabels([c[:15] for c in sorted(unique_classes)], rotation=45, ha='right', fontsize=7)
    ax.set_title(feat[:40], fontsize=9, color='#e0e0e0')
    ax.set_ylabel('Value (clipped 99th pct)', fontsize=8)

plt.tight_layout()
plt.savefig('../docs/eda_plots/05_attack_feature_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved → ../docs/eda_plots/05_attack_feature_boxplots.png')

## 10. Key EDA Findings & Next Steps

> This section summarizes every finding from the EDA and maps each issue to a concrete Phase 2 action.

In [ ]:
print('=' * 65)
print('EDA SUMMARY — CICIDS-2017')
print('=' * 65)
print(f'''
1. DATASET SHAPE
   ✔ {df.shape[0]:,} rows × {df.shape[1]} columns
   ✔ {len(numeric_cols)} numeric features + 1 label column

2. CLASS IMBALANCE
   ✔ BENIGN class heavily dominates (>80% of data)
   ✔ Infiltration class is the rarest (often < 100 samples)
   → Phase 2 Fix: SMOTE on training set only

3. DATA QUALITY ISSUES
   ✔ Inf values found in flow-rate features
   ✔ NaN values present in some columns
   ✔ Zero-variance features identified
   → Phase 2 Fix: Replace Inf, drop NaN rows, drop zero-var cols

4. FEATURE CORRELATIONS
   ✔ Many highly correlated pairs (|r| > 0.90) identified
   → We keep all features for maximum SHAP expressiveness

5. TOP DISCRIMINATIVE FEATURES (EDA Quick Pass)
''')
for i, (feat, imp) in enumerate(top_features.head(4).items(), 1):
    print(f'     {i}. {feat} (importance: {imp:.5f})')
print(f'''
6. NEXT STEPS — PHASE 2
   □ Implement preprocessing pipeline
   □ Apply SMOTE on training split only
   □ Encode labels with LabelEncoder
   □ Save processed train.csv, test.csv
   □ Save scaler and encoder artifacts
   Notebook: notebooks/02_preprocessing.ipynb
''')
print('=' * 65)

In [ ]:
import json

eda_meta = {
    'label_col': LABEL_COL,
    'feature_cols': FEATURE_COLS,
    'numeric_cols': numeric_cols,
    'n_classes': int(df[LABEL_COL].nunique()),
    'class_names': sorted(df[LABEL_COL].unique().tolist()),
    'zero_variance_cols': zero_var.index.tolist(),
    'inf_cols': list(inf_counts.keys()),
    'null_cols': null_cols.index.tolist(),
    'top_20_features': top_features.index.tolist(),
    'total_samples': len(df),
    'class_counts': df[LABEL_COL].value_counts().to_dict()
}

os.makedirs('../data/processed', exist_ok=True)
meta_path = '../data/processed/eda_meta.json'
with open(meta_path, 'w') as f:
    json.dump(eda_meta, f, indent=2)

print(f'✅ EDA metadata saved → {meta_path}')
print(f'   This file will be loaded by notebooks/02_preprocessing.ipynb')